# Failure analysis after the Final test

**CRISP-DM stage:** Evaluation

Half B showed that both locked recommendations failed at their locked lines. This notebook asks **why**, with two measurements. Both are **after-the-fact analyses**: nothing locked is changed, no new threshold or winner is chosen, and the results are reported as they come out, even if they do not support a suspected cause.

1. **Did the threshold carry over to new good parts?** For every configuration, the share of good photos above its locked cost-based line, on half A (where the line was chosen) and on half B, also broken down by good part.
2. **Are the missed defects the small ones?** Each defective half B photo's defect area is measured from its mask in the original image. Defects are split into thirds by size (small, medium, large) per product, and the share caught at the locked line is compared across the thirds. A rank correlation between defect area and score is added.

**Data:** saved photo scores (`outputs/results/05_core`, `07_core`, `final_test/half_b`), locked lines (`07_evaluation/cost_by_run.csv`), and the public test masks of MVTec AD 2 in `data/demo_photos/` (not in git; CC BY-NC-SA 4.0).

**Runs on:** the laptop, no GPU.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy
from PIL import Image
from scipy.stats import spearmanr

PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "src").exists():
    PROJECT_FOLDER = PROJECT_FOLDER.parent
RESULTS = PROJECT_FOLDER / "outputs/results"
OUTPUT_FOLDER = RESULTS / "final_test/failure_analysis"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
FIGURE_FOLDER = PROJECT_FOLDER / "outputs/figures"
MASK_FOLDER = PROJECT_FOLDER / "data/demo_photos"
PRODUCTS = ["sheet_metal", "vial"]
CONFIGS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8"]
# The locked recommendation per product, highlighted in the charts
LOCKED_CHOICE = {"sheet_metal": "C6", "vial": "C7"}
# Deck palette: one accent for the locked choice, grey for the alternatives
ACCENT = "#2A6F77"
GREY = "#B3B3B3"
DARK = "#333333"
MUTED = "#8C8C8C"


def read_scores(score_file):
    """Return file names, labels (True for defective) and scores from one run's file."""
    names = []
    labels = []
    scores = []
    with open(score_file, encoding="utf-8") as file:
        for row in csv.DictReader(file):
            names.append(row["file_name"])
            labels.append(row["label"] == "bad")
            scores.append(float(row["score"]))
    return names, numpy.array(labels), numpy.array(scores)


# Locked cost-based lines at the main setting: 1 percent defects, cost ratio 10
locked_lines = {}
with open(RESULTS / "07_evaluation/cost_by_run.csv", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if float(row["defect_rate"]) == 0.01 and float(row["cost_ratio"]) == 10:
            locked_lines[(row["config"], row["product"], row["seed"])] = float(row["cheapest_threshold"])

# Every run's half A and half B score files, by configuration, product and seed
runs = []
for config_id in CONFIGS:
    for product in PRODUCTS:
        for half_b_file in sorted((RESULTS / "final_test/half_b" / config_id).glob(config_id + "_" + product + "_seed*_halfB_photo_scores.csv")):
            seed = half_b_file.name.split("_seed")[1].split("_")[0]
            half_a_file = None
            for folder in [RESULTS / "05_core", RESULTS / "07_core"]:
                candidate = folder / config_id / (config_id + "_" + product + "_seed" + seed + "_photo_scores.csv")
                if candidate.exists():
                    half_a_file = candidate
            runs.append({"config": config_id, "product": product, "seed": seed, "half_a": half_a_file,
                         "half_b": half_b_file, "line": locked_lines[(config_id, product, seed)]})
print(len(runs), "runs read")

## 1. Did the locked line carry over to new good parts?

The cost-based line was chosen on half A and sits just above its highest-scoring good photo, so it rejects no good half A photo by construction. The question is how many **new** good parts (half B) score above it.

In [ ]:
rows = []
part_rows = []
for run in runs:
    shares = {}
    for half in ["half_a", "half_b"]:
        names, labels, scores = read_scores(run[half])
        good = ~labels
        shares[half] = float((scores[good] > run["line"]).mean())
        # The same share for each good part, so one unusual part can be seen
        parts = sorted(set(name.split("_")[0] for name, is_bad in zip(names, labels) if not is_bad))
        for part in parts:
            chosen = numpy.array([(name.split("_")[0] == part) and not is_bad for name, is_bad in zip(names, labels)])
            part_rows.append({"config": run["config"], "product": run["product"], "seed": run["seed"],
                              "half": half[-1].upper(), "good_part": part, "photos": int(chosen.sum()),
                              "share_above_locked_line": round(float((scores[chosen] > run["line"]).mean()), 4)})
    rows.append({"config": run["config"], "product": run["product"], "seed": run["seed"],
                 "good_rejected_half_a": round(shares["half_a"], 4), "good_rejected_half_b": round(shares["half_b"], 4)})

# Mean over seeds per configuration and product
summary = []
for product in PRODUCTS:
    for config_id in CONFIGS:
        chosen = [row for row in rows if row["product"] == product and row["config"] == config_id]
        summary.append({"product": product, "config": config_id, "seeds": len(chosen),
                        "good_rejected_half_a": round(sum(r["good_rejected_half_a"] for r in chosen) / len(chosen), 4),
                        "good_rejected_half_b": round(sum(r["good_rejected_half_b"] for r in chosen) / len(chosen), 4)})

for name, table in [("threshold_transfer.csv", summary), ("threshold_transfer_by_part.csv", part_rows)]:
    with open(OUTPUT_FOLDER / name, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(table[0].keys()))
        writer.writeheader()
        for row in table:
            writer.writerow(row)
for row in summary:
    print(row["product"].ljust(12), row["config"], "good photos rejected: half A", row["good_rejected_half_a"],
          "-> half B", row["good_rejected_half_b"])
print()
for row in part_rows:
    if row["config"] == LOCKED_CHOICE[row["product"]] and row["seed"] == "42":
        print(row)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True)
for column in range(len(PRODUCTS)):
    product = PRODUCTS[column]
    axis = axes[column]
    for position in range(len(CONFIGS)):
        config_id = CONFIGS[position]
        row = [r for r in summary if r["product"] == product and r["config"] == config_id][0]
        colour = ACCENT if config_id == LOCKED_CHOICE[product] else GREY
        y = len(CONFIGS) - 1 - position
        axis.plot([row["good_rejected_half_a"], row["good_rejected_half_b"]], [y, y], color=colour, linewidth=2.5)
        axis.scatter([row["good_rejected_half_a"]], [y], color="white", edgecolor=colour, s=70, zorder=3, linewidth=2)
        axis.scatter([row["good_rejected_half_b"]], [y], color=colour, s=70, zorder=3)
        axis.text(row["good_rejected_half_b"] + 0.03, y, str(round(row["good_rejected_half_b"] * 100)) + "%",
                  va="center", fontsize=9, color=DARK if colour == ACCENT else MUTED)
    axis.set_yticks(range(len(CONFIGS)))
    axis.set_yticklabels(list(reversed(CONFIGS)))
    axis.set_xlim(-0.03, 1.12)
    axis.set_title(product.replace("_", " ").title(), color=DARK)
    axis.set_xlabel("Share of good photos rejected at the locked line", color=DARK)
    for side in ["top", "right"]:
        axis.spines[side].set_visible(False)
    axis.grid(axis="x", alpha=0.3)
figure.suptitle("Open circle: half A (where the line was set). Filled: half B (new good parts)", color=MUTED, fontsize=10)
figure.tight_layout()
figure.savefig(FIGURE_FOLDER / "failure_threshold_transfer.png", dpi=200)
plt.close(figure)
print("Saved the chart")

## 2. Are the missed defects the small ones?

The defect area is the number of defect pixels in the mask, in the original photo. Defective half B photos are split into thirds by area per product. A defect counts as caught when the photo scores above the locked cost-based line. The rank correlation (Spearman) between area and score shows whether bigger defects score higher, without needing a line.

In [ ]:
areas = {}
for product in PRODUCTS:
    mask_folder = MASK_FOLDER / product / "test_public/ground_truth/bad"
    if not mask_folder.exists():
        raise FileNotFoundError("Masks not found in " + str(mask_folder))
    for mask_file in mask_folder.glob("*_mask.png"):
        mask = numpy.array(Image.open(mask_file))
        areas[(product, mask_file.name.replace("_mask.png", ".png"))] = int((mask > 0).sum())

size_rows = []
correlation_rows = []
for product in PRODUCTS:
    # Thirds of the defect areas of this product's defective half B photos
    names, labels, scores = read_scores(runs[[r["product"] for r in runs].index(product)]["half_b"])
    defective_names = [name for name, is_bad in zip(names, labels) if is_bad]
    product_areas = numpy.array([areas[(product, name)] for name in defective_names])
    cut_low, cut_high = numpy.percentile(product_areas, [100 / 3, 200 / 3])
    print(product, ":", len(defective_names), "defective photos, area thirds cut at", int(cut_low), "and", int(cut_high), "pixels")

    def size_class(area):
        if area <= cut_low:
            return "small"
        if area <= cut_high:
            return "medium"
        return "large"

    for config_id in CONFIGS:
        caught = {"small": [], "medium": [], "large": []}
        correlations = []
        for run in runs:
            if run["product"] != product or run["config"] != config_id:
                continue
            names, labels, scores = read_scores(run["half_b"])
            run_areas = []
            run_scores = []
            for name, is_bad, score in zip(names, labels, scores):
                if is_bad:
                    caught[size_class(areas[(product, name)])].append(score > run["line"])
                    run_areas.append(areas[(product, name)])
                    run_scores.append(score)
            correlations.append(spearmanr(run_areas, run_scores).statistic)
        row = {"product": product, "config": config_id}
        for size in ["small", "medium", "large"]:
            row["caught_" + size] = round(float(numpy.mean(caught[size])), 4)
        size_rows.append(row)
        correlation_rows.append({"product": product, "config": config_id,
                                 "spearman_area_vs_score": round(float(numpy.mean(correlations)), 3)})

with open(OUTPUT_FOLDER / "defect_size_vs_caught.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(size_rows[0].keys()) + ["spearman_area_vs_score"])
    writer.writeheader()
    for row, correlation in zip(size_rows, correlation_rows):
        merged = dict(row)
        merged["spearman_area_vs_score"] = correlation["spearman_area_vs_score"]
        writer.writerow(merged)
for row, correlation in zip(size_rows, correlation_rows):
    print(row["product"].ljust(12), row["config"], "caught small", row["caught_small"], "medium", row["caught_medium"],
          "large", row["caught_large"], "| Spearman", correlation["spearman_area_vs_score"])

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
sizes = ["small", "medium", "large"]
for column in range(len(PRODUCTS)):
    product = PRODUCTS[column]
    axis = axes[column]
    # End labels sorted by height and pushed at least 0.05 apart, so they never overlap
    ends = []
    for config_id in CONFIGS:
        row = [r for r in size_rows if r["product"] == product and r["config"] == config_id][0]
        ends.append((row["caught_large"], config_id))
    ends.sort()
    label_height = {}
    last = None
    for value, config_id in ends:
        height = value
        if last is not None and height < last + 0.05:
            height = last + 0.05
        label_height[config_id] = height
        last = height
    for config_id in CONFIGS:
        row = [r for r in size_rows if r["product"] == product and r["config"] == config_id][0]
        values = [row["caught_" + size] for size in sizes]
        is_choice = config_id == LOCKED_CHOICE[product]
        axis.plot(range(3), values, marker="o", color=ACCENT if is_choice else GREY,
                  linewidth=3 if is_choice else 1.2, zorder=3 if is_choice else 2)
        axis.text(2.1, label_height[config_id], config_id, va="center", fontsize=8, color=DARK if is_choice else MUTED)
    axis.set_xticks(range(3))
    axis.set_xticklabels(["smallest third", "middle third", "largest third"])
    axis.set_xlim(-0.2, 2.35)
    axis.set_ylim(-0.03, 1.03)
    axis.set_title(product.replace("_", " ").title(), color=DARK)
    axis.set_xlabel("Defect size (area in the mask)", color=DARK)
    for side in ["top", "right"]:
        axis.spines[side].set_visible(False)
    axis.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Share of defective photos caught at the locked line", color=DARK)
figure.tight_layout()
figure.savefig(FIGURE_FOLDER / "failure_defect_size.png", dpi=200)
plt.close(figure)
print("Saved the chart")